#Dataset sintético para detección de dolor facial

In [ ]:
!pip install -q google-genai google-cloud-storage tqdm

from google.colab import auth
auth.authenticate_user()

## Configuración

In [ ]:
import os

# Sustituir por un proyecto propio de Google Cloud (con Vertex AI habilitado)
# y un bucket de Cloud Storage de ese proyecto
PROJECT_ID = "tu-proyecto-gcp"
GCS_BUCKET = "gs://tu-bucket/carpeta"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud config set project {PROJECT_ID} --quiet
print(f"Proyecto: {PROJECT_ID}")
print(f"Bucket: {GCS_BUCKET}")

## Consultas

In [ ]:
from itertools import product as cartesian_product
import random

REPEATS = 1
OUTPUT_DIR = "/content/pain_dataset"
DURATION = 4

#NIVELES DE DOLOR

PAIN_EXPRESSIONS = {
    "none": (
        "a relaxed, neutral expression: "
        "smooth brow, eyes gently closed, mouth naturally closed. "
        "Natural movements"
    ),
    "moderate": (
        "a moderate physical discomfort: "
        "slightly furrowed brow, moderately tightened closed eyelids, wrinkled nose, "
        "mouth firmly closed with pressed lips. Visible annoyance and discomfort"
    ),
    "severe": (
        "an intensely strained expression: "
        "deep furrows between eyebrows, tightly shut eyes, "
        "wrinkled nose, tightly clenched teeth, "
        "occasionally raised upper lip. Visible severe physical distress"
    ),
}


#TIPOS DE VIDEO

VIDEO_TYPES = [
    {"id": "const_none",     "type": "constant", "from": "none",     "to": "none",     "label": 0},
    {"id": "const_moderate", "type": "constant", "from": "moderate", "to": "moderate", "label": 1},
    {"id": "const_severe",   "type": "constant", "from": "severe",   "to": "severe",   "label": 2},
]


#DEMOGRAFÍA

GENDERS = ["man", "woman"]

AGES = [
    "young adult (20-35 years old)",
    "middle-aged (45-60 years old)",
    "elderly (70-85 years old)",
]

ETHNICITIES = [
    "white skin tone",
    "Latin American features",
    "dark skin",
    "North African features",
    "East Asian features",
    "South Asian features",
]

DEMOGRAPHIC_PROFILES = [
    {"gender": g, "age": a, "ethnicity": e}
    for g, a, e in cartesian_product(GENDERS, AGES, ETHNICITIES)
]

#VARIACIONES

HAIR_LENGTHS = ["short", "medium-length", "long", "bald"]
HAIR_TEXTURES = ["straight", "wavy", "curly"]
HAIR_COLORS = ["black", "brown", "blonde", "white"]

FACE_SHAPES = [
    "round face",
    "oval face",
    "square face",
    "long narrow face",
    "diamond-shaped face",
]

FACE_DETAILS = [
    "prominent nose",
    "small nose",
    "thick eyebrows",
    "thin eyebrows",
    "wide forehead",
    "narrow forehead",
    "thin lips",
    "full lips",
    "freckles",
    "acne",
    "full cheeks",
    "sunken cheeks",
    "beard shadow",
]

#ESCENA

LIGHTING_STYLES = [
    "Soft diffused lighting, minimal shadows. ",
    "Bright clinical lighting with sharp shadows. ",
    "Top-down directional lighting with sharp shadows. ",
    "Warm surgical lighting. ",
    "Slightly dim ambient lighting. ",
    "Natural window daylight mixed with ceiling lights. ",
    ""
]

SHARPNESS_STYLES = [
    "Sharp focus with every skin detail visible. ",
    "Sharp focus with slight natural softness. ",
    "High-definition detail with fine texture. ",
    "Slightly soft focus, hospital camera feel. ",
    "",
]

SKIN_TEXTURE_STYLES = [
    "Natural skin texture. ",
    "Dry matte skin texture. ",
    "Smooth skin with subtle imperfections. ",
    "Weathered skin  with natural creases and lines. ",
    "Skin with visible veins and natural discoloration. ",
    "Slightly flushed skin, redness on cheeks. ",
    ""
]

def generate_appearance():
    """Genera una descripción aleatoria de apariencia física."""
    hair_length = random.choice(HAIR_LENGTHS)
    if hair_length == "bald":
        hair_desc = "bald"
    else:
        texture = random.choice(HAIR_TEXTURES)
        color = random.choice(HAIR_COLORS)
        hair_desc = f"{hair_length} {texture} {color} hair"

    face_shape = random.choice(FACE_SHAPES)
    face_detail = random.choice(FACE_DETAILS)

    return {
        "hair": hair_desc,
        "face_shape": face_shape,
        "face_details": face_detail,
        "full_description": f"{face_shape}, {face_detail}, {hair_desc}",
    }


def generate_scene_variation():
    """Genera una variación aleatoria de la escena."""
    return {
        "lighting": random.choice(LIGHTING_STYLES),
        "sharpness": random.choice(SHARPNESS_STYLES),
        "skin_texture": random.choice(SKIN_TEXTURE_STYLES),
    }


#PROMPTS

NEGATIVE_PROMPT = (
    "side view, profile view, partial face, cropped face, medical equipment, "
    "camera movement, zoom, panning, tilting, camera tracking, camera drift, "
    "cartoon, anime, illustration, watermark, text, logo, talking, shouting, "
    "deformed, glasses, hat, makeup, accessories, jewelry"
)


def build_prompt(video_type, profile, appearance, scene):
    g = profile["gender"]
    a = profile["age"]
    e = profile["ethnicity"]
    app = appearance["full_description"]

    expr = PAIN_EXPRESSIONS[video_type["from"]]

    return (
        f"A strictly static, locked-off vertical overhead shot from a "
        f"ceiling-mounted camera, zero camera movement, zero optical zoom. "
        f"A {a} {g} with {e}, {app}, "
        f"lying face-up on a hospital bed. "
        f"Fixed frame including the pillow edges and displaying the full face "
        f"with generous margins. "
        f"The person displays {expr} "
        f"consistently throughout the entire video. "
        f"Clinical medical setting. "
        f"{scene['lighting']}{scene['sharpness']}"
        f"{scene['skin_texture']}"
    )

#PRUEBA
sample_appearance = generate_appearance()
sample_scene = generate_scene_variation()
sample_prompt = build_prompt(VIDEO_TYPES[1], DEMOGRAPHIC_PROFILES[7], sample_appearance, sample_scene)
print(f"\nEjemplo de prompt ({len(sample_prompt.split())} palabras, {len(sample_prompt)} chars):")
print(sample_prompt)


Ejemplo de prompt (106 palabras, 759 chars):
A strictly static, locked-off vertical overhead shot from a ceiling-mounted camera, zero camera movement, zero optical zoom. A middle-aged (45-60 years old) man with Latin American features, oval face, narrow forehead, short wavy black hair, lying face-up on a hospital bed. Fixed frame including the pillow edges and displaying the full face with generous margins. The person displays a moderate physical discomfort: slightly furrowed brow, moderately tightened closed eyelids, wrinkled nose, mouth firmly closed with pressed lips. Visible annoyance and discomfort consistently throughout the entire video. Clinical medical setting. Soft diffused lighting, minimal shadows. Sharp focus with slight natural softness. Slightly flushed skin, redness on cheeks. 


## Generar plan de vídeos

In [ ]:
import random
import time

random.seed(int(time.time()))

video_plan = []
vid_counter = 0

for rep in range(REPEATS):
    for vtype in VIDEO_TYPES:
        for profile in DEMOGRAPHIC_PROFILES:
            video_id = f"v{vid_counter:05d}"

            appearance = generate_appearance()
            scene = generate_scene_variation()
            prompt = build_prompt(vtype, profile, appearance, scene)

            video_plan.append({
                "video_id": video_id,
                "label": vtype["label"],
                "prompt": prompt,
            })
            vid_counter += 1

random.shuffle(video_plan)


## Inicializar Veo 3

In [ ]:
from google import genai
from google.genai.types import GenerateVideosConfig

client = genai.Client()

VEO_MODEL = "veo-3.1-fast-generate-001"

## Generar base de datos

In [ ]:
import time
import json
import os
from tqdm.notebook import tqdm
from google.cloud import storage as gcs

for label in [0, 1, 2]:
    os.makedirs(f"{OUTPUT_DIR}/videos/{label}", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/annotations", exist_ok=True)

gcs_client = gcs.Client(project=PROJECT_ID)

#Detectar videos ya generados
already_done = set()
for label in [0, 1, 2]:
    label_dir = f"{OUTPUT_DIR}/videos/{label}"
    for f in os.listdir(label_dir):
        if f.endswith(".mp4"):
            already_done.add(f.replace(".mp4", ""))
if already_done:
    print(f"Reanudando: {len(already_done)} videos ya generados")

success = len(already_done)
errors = []
total = len(video_plan)

for i, entry in enumerate(tqdm(video_plan, desc="Generando dataset")):
    vid_id = entry["video_id"]
    label = entry["label"]

    if vid_id in already_done:
        continue

    video_gcs_uri = f"{GCS_BUCKET}/videos/{vid_id}/"

    try:
        operation = client.models.generate_videos(
            model=VEO_MODEL,
            prompt=entry["prompt"],
            config=GenerateVideosConfig(
                aspect_ratio="16:9",
                resolution="720p",
                duration_seconds=DURATION,
                generate_audio=False,
                person_generation="allow_all",
                negative_prompt=NEGATIVE_PROMPT,
                output_gcs_uri=video_gcs_uri,
            ),
        )

        while not operation.done:
            time.sleep(10)
            operation = client.operations.get(operation)

        if operation.result and operation.result.generated_videos:
            gcs_uri = operation.result.generated_videos[0].video.uri

            #Descargar video en la carpeta de su label
            blob_parts = gcs_uri.replace("gs://", "").split("/", 1)
            blob = gcs_client.bucket(blob_parts[0]).blob(blob_parts[1])
            local_video = f"{OUTPUT_DIR}/videos/{label}/{vid_id}.mp4"
            blob.download_to_filename(local_video)

            success += 1

            if success % 10 == 0:
                print(f"{success}/{total} {vid_id} --> carpeta {label}/")
        else:
            print(f"{vid_id}: sin output")
            print(f"Result: {operation.result}")
            errors.append({"video_id": vid_id, "error": "no_output", "details": str(operation), "entry": entry})

    except Exception as e:
        errors.append({"video_id": vid_id, "error": str(e), "entry": entry})
        print(f"{vid_id}: {str(e)[:80]}")
        time.sleep(30)
        continue

print(f"\n")
print(f"Generación completada")
print(f"Exitosos: {success}/{total}")
print(f"Errores:  {len(errors)}")


## Descargar dataset

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/pain_dataset_veo3", "zip", OUTPUT_DIR)
zip_size = os.path.getsize("/content/pain_dataset_veo3.zip") / (1024 * 1024)
files.download("/content/pain_dataset_veo3.zip")